In [1]:
import os
import json
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import random

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from sklearn.decomposition import PCA

In [2]:
import os

os.makedirs("Models", exist_ok=True)

print("Models folder is ready.")

Models folder is ready.


In [3]:
############################################################
# CONFIG
############################################################

TRAIN_DIR = "/kaggle/input/datasets/vikas635233/dataset-for-train-test/EMG-EPN612 Dataset/trainingJSON"
TEST_DIR = "/kaggle/input/datasets/vikas635233/dataset-for-train-test/EMG-EPN612 Dataset/testingJSON"

TARGET_LEN = 1019
PCA_COMPONENTS = 6

BATCH_SIZE = 32
EPOCHS = 200
LR = 1e-3

############################################################
# LABEL MAP
############################################################

LABEL_MAP = {
    "noGesture": 0,
    "fist": 1,
    "waveIn": 2,
    "waveOut": 3,
    "open": 4,
    "pinch": 5
}

############################################################
# PAD / CROP
############################################################

def pad_or_crop(signal, target_len=TARGET_LEN):

    current_len = signal.shape[1]

    if current_len > target_len:

        signal = signal[:, :target_len]

    elif current_len < target_len:

        pad = target_len - current_len

        signal = np.pad(
            signal,
            ((0, 0), (0, pad)),
            mode='constant'
        )

    return signal

############################################################
# LOAD DATA
############################################################

def load_dataset(folder, allowed_users=None):



    X = []
    y = []

    for root, dirs, files in os.walk(folder):

 

        user_name = os.path.basename(root)

        if allowed_users is not None:
            if user_name not in allowed_users:
               
                continue

        for file in files:

            if not file.endswith(".json"):
                continue

            path = os.path.join(root, file)

        

            with open(path, "r") as f:
                data = json.load(f)

         

            if "trainingSamples" not in data:
               
                continue

            samples = data["trainingSamples"]

           
            for key in samples:

                sample = samples[key]

                gesture = sample["gestureName"]

                if gesture not in LABEL_MAP:
                    
                    continue

                emg = sample["emg"]

                signal = np.array([
                    emg["ch1"],
                    emg["ch2"],
                    emg["ch3"],
                    emg["ch4"],
                    emg["ch5"],
                    emg["ch6"],
                    emg["ch7"],
                    emg["ch8"]
                ], dtype=np.float32)

                signal = pad_or_crop(signal)

                X.append(signal)
                y.append(LABEL_MAP[gesture])


    return np.array(X, dtype=np.float32), np.array(y)

############################################################
# PCA
############################################################

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def apply_pca(X_train, X_test):

    scaler = StandardScaler()

    pca = PCA(
        n_components=PCA_COMPONENTS
    )

    temp = X_train.transpose(0,2,1)
    temp = temp.reshape(-1,8)

    temp = scaler.fit_transform(temp)

    pca.fit(temp)

    train_out = []

    for sample in X_train:

        sample = sample.T
        sample = scaler.transform(sample)
        sample = pca.transform(sample)
        sample = sample.T

        train_out.append(sample)

    test_out = []

    for sample in X_test:

        sample = sample.T
        sample = scaler.transform(sample)
        sample = pca.transform(sample)
        sample = sample.T

        test_out.append(sample)

    return (
        np.array(train_out, dtype=np.float32),
        np.array(test_out, dtype=np.float32)
    )

In [4]:
############################################################
# DATASET
############################################################

class EMGDataset(Dataset):

    def __init__(self, X, y):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

############################################################
# CNN MODEL
############################################################

In [5]:
############################################################
# CNN MODEL
############################################################

class PCA_CNN(nn.Module):

    def __init__(self):

        super().__init__()

        ############################################################
        # Depthwise + Pointwise Conv1
        ############################################################

        self.dw1 = nn.Conv1d(
            in_channels=6,
            out_channels=6,
            kernel_size=2,
            groups=6
        )

        self.pw1 = nn.Conv1d(
            in_channels=6,
            out_channels=32,
            kernel_size=1
        )

        self.pool = nn.MaxPool1d(
            kernel_size=2
        )

        ############################################################
        # Depthwise + Pointwise Conv2
        ############################################################

        self.dw2 = nn.Conv1d(
            in_channels=32,
            out_channels=32,
            kernel_size=2,
            groups=32
        )

        self.pw2 = nn.Conv1d(
            in_channels=32,
            out_channels=32,
            kernel_size=1
        )

        ############################################################
        # GAP
        ############################################################

        self.gap = nn.AdaptiveAvgPool1d(1)

        ############################################################
        # FC
        ############################################################

        self.fc1 = nn.Linear(
            32,
            64
        )

        self.fc2 = nn.Linear(
            64,
            6
        )

    def forward(self, x):

        x = F.relu(self.pw1(self.dw1(x)))

        x = self.pool(x)

        x = F.relu(self.pw2(self.dw2(x)))

        x = self.gap(x)

        x = x.squeeze(-1)

        x = F.relu(self.fc1(x))

        x = self.fc2(x)

        return x

In [6]:
model = PCA_CNN()

print(model)

print("Parameters:",
      sum(p.numel() for p in model.parameters()))

PCA_CNN(
  (dw1): Conv1d(6, 6, kernel_size=(2,), stride=(1,), groups=6)
  (pw1): Conv1d(6, 32, kernel_size=(1,), stride=(1,))
  (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dw2): Conv1d(32, 32, kernel_size=(2,), stride=(1,), groups=32)
  (pw2): Conv1d(32, 32, kernel_size=(1,), stride=(1,))
  (gap): AdaptiveAvgPool1d(output_size=1)
  (fc1): Linear(in_features=32, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=6, bias=True)
)
Parameters: 3896


In [7]:
############################################################
# LOAD DATA
############################################################
TEST_USERS = [
    "user1","user2","user3","user4","user5",
    "user6","user7","user8","user9","user10",
    "user11","user12","user13","user14","user15",
    "user16","user17","user18","user19","user20",
    "user21"
]
print("Loading data...")

print("TRAIN_DIR =", TRAIN_DIR)
print("TEST_DIR  =", TEST_DIR)

X_train, y_train = load_dataset(TRAIN_DIR)

X_test, y_test = load_dataset(
    TEST_DIR,
    TEST_USERS
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

print("Train distribution:")
print(np.unique(y_train, return_counts=True))

print("Test distribution:")
print(np.unique(y_test, return_counts=True))

############################################################
# PCA
############################################################

print("Applying PCA...")

X_train, X_test = apply_pca(
    X_train,
    X_test
)

np.save("X_train_pca.npy", X_train)
np.save("y_train.npy", y_train)

np.save("X_test_pca.npy", X_test)
np.save("y_test.npy", y_test)

print("After PCA")

print("Train :", X_train.shape)
print("Test  :", X_test.shape)

############################################################
# DATALOADER
############################################################

train_dataset = EMGDataset(
    X_train,
    y_train
)

test_dataset = EMGDataset(
    X_test,
    y_test
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

Loading data...
TRAIN_DIR = /kaggle/input/datasets/vikas635233/dataset-for-train-test/EMG-EPN612 Dataset/trainingJSON
TEST_DIR  = /kaggle/input/datasets/vikas635233/dataset-for-train-test/EMG-EPN612 Dataset/testingJSON
Train shape: (45900, 8, 1019)
Test shape : (3150, 8, 1019)
X_train: (45900, 8, 1019)
y_train: (45900,)
X_test : (3150, 8, 1019)
y_test : (3150,)
Train distribution:
(array([0, 1, 2, 3, 4, 5]), array([7650, 7650, 7650, 7650, 7650, 7650]))
Test distribution:
(array([0, 1, 2, 3, 4, 5]), array([525, 525, 525, 525, 525, 525]))
Applying PCA...
After PCA
Train : (45900, 6, 1019)
Test  : (3150, 6, 1019)


In [8]:
############################################################
# DEVICE
############################################################

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

############################################################
# MODEL
############################################################

model = PCA_CNN().to(device)

total_params = sum(p.numel() for p in model.parameters())
print("Total Parameters:", total_params)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR
)

############################################################
# TRAIN
############################################################

best_acc = 0

for epoch in range(EPOCHS):

    ########################################################
    # TRAIN
    ########################################################

    model.train()

    train_loss = 0
    train_correct = 0
    train_total = 0

    for x, y in train_loader:

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        outputs = model(x)

        loss = criterion(
            outputs,
            y
        )

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

        preds = outputs.argmax(1)

        train_total += y.size(0)

        train_correct += (
            preds == y
        ).sum().item()

    train_acc = (
        100.0 *
        train_correct /
        train_total
    )

    ########################################################
    # TEST
    ########################################################

    model.eval()

    test_correct = 0
    test_total = 0

    with torch.no_grad():

        for x, y in test_loader:

            x = x.to(device)
            y = y.to(device)

            outputs = model(x)

            preds = outputs.argmax(1)

            test_total += y.size(0)

            test_correct += (
                preds == y
            ).sum().item()

    test_acc = (
        100.0 *
        test_correct /
        test_total
    )

    ########################################################
    # PRINT RESULTS
    ########################################################

    print(
        f"Epoch {epoch+1:03d} | "
        f"Loss {train_loss:.4f} | "
        f"Train Acc {train_acc:.2f}% | "
        f"Test Acc {test_acc:.2f}%"
    )

    ########################################################
    # SAVE BEST MODEL
    ########################################################

    if test_acc > best_acc:

        best_acc = test_acc

        torch.save(
            model.state_dict(),
            "Models/best_model_depthwise.pth"
        )

        print(
            f"Saved model : {best_acc:.2f}%"
        )

print("\nTraining Complete")
print("Best Test Accuracy =", best_acc)

Device: cuda
Total Parameters: 3896
Epoch 001 | Loss 1488.8491 | Train Acc 57.78% | Test Acc 76.83%
Saved model : 76.83%
Epoch 002 | Loss 926.3200 | Train Acc 76.22% | Test Acc 81.71%
Saved model : 81.71%
Epoch 003 | Loss 720.0657 | Train Acc 82.13% | Test Acc 86.98%
Saved model : 86.98%
Epoch 004 | Loss 610.5025 | Train Acc 84.72% | Test Acc 88.98%
Saved model : 88.98%
Epoch 005 | Loss 560.1275 | Train Acc 85.88% | Test Acc 88.83%
Epoch 006 | Loss 526.2513 | Train Acc 86.61% | Test Acc 90.44%
Saved model : 90.44%
Epoch 007 | Loss 496.4450 | Train Acc 87.24% | Test Acc 90.29%
Epoch 008 | Loss 476.4637 | Train Acc 87.83% | Test Acc 90.32%
Epoch 009 | Loss 458.7675 | Train Acc 88.44% | Test Acc 90.73%
Saved model : 90.73%
Epoch 010 | Loss 440.1859 | Train Acc 88.83% | Test Acc 90.19%
Epoch 011 | Loss 431.1744 | Train Acc 89.08% | Test Acc 90.70%
Epoch 012 | Loss 416.9148 | Train Acc 89.42% | Test Acc 90.86%
Saved model : 90.86%
Epoch 013 | Loss 407.7172 | Train Acc 89.74% | Test Acc 90.7

In [9]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

model.load_state_dict(
    torch.load("Models/best_model_depthwise.pth")
)

model.eval()

y_true = []
y_pred = []

with torch.no_grad():

    for x, y in test_loader:

        x = x.to(device)

        outputs = model(x)

        preds = outputs.argmax(1)

        y_true.extend(y.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

print("Confusion Matrix")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report")
print(classification_report(y_true, y_pred))

Confusion Matrix
[[522   1   0   0   0   2]
 [  0 497   2   0   4  22]
 [  0   2 514   2   0   7]
 [  2   0   7 470  40   6]
 [  0  10   2  23 452  38]
 [  1   8   1   2  22 491]]

Classification Report
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       525
           1       0.96      0.95      0.95       525
           2       0.98      0.98      0.98       525
           3       0.95      0.90      0.92       525
           4       0.87      0.86      0.87       525
           5       0.87      0.94      0.90       525

    accuracy                           0.94      3150
   macro avg       0.94      0.94      0.94      3150
weighted avg       0.94      0.94      0.94      3150



In [10]:
############################################################
# WEIGHT QUANTIZATION
############################################################
def quantize_weights(weights, bits):

    qmin = -(2 ** (bits - 1))
    qmax = (2 ** (bits - 1)) - 1

    # Linear layer
    if weights.ndim == 2:

        out = torch.empty_like(weights)

        for i in range(weights.size(0)):

            w = weights[i]

            max_val = w.abs().max()

            if max_val == 0:
                out[i] = w
                continue

            scale = max_val / qmax

            q = torch.round(w / scale)
            q = torch.clamp(q, qmin, qmax)

            out[i] = q * scale

        return out

    # Conv1D layer
    elif weights.ndim == 3:

        out = torch.empty_like(weights)

        for i in range(weights.size(0)):

            w = weights[i]

            max_val = w.abs().max()

            if max_val == 0:
                out[i] = w
                continue

            scale = max_val / qmax

            q = torch.round(w / scale)
            q = torch.clamp(q, qmin, qmax)

            out[i] = q * scale

        return out

    # Bias or scalar
    else:

        max_val = weights.abs().max()

        if max_val == 0:
            return weights.clone()

        scale = max_val / qmax

        q = torch.round(weights / scale)
        q = torch.clamp(q, qmin, qmax)

        return q * scale

In [11]:
############################################################
# UNIFORM 8-BIT QUANTIZATION
############################################################

import copy
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

############################################################
# LOAD BEST MODEL
############################################################

base_model = PCA_CNN().to(device)

base_model.load_state_dict(
    torch.load("Models/best_model_depthwise.pth")
)

base_model.eval()

############################################################
# CREATE UNIFORM 8-BIT MODEL
############################################################

uniform8_model = copy.deepcopy(base_model)

with torch.no_grad():

    for name, param in uniform8_model.named_parameters():

        if "weight" in name:

            param.data = quantize_weights(
                param.data,
                bits=8
            )

############################################################
# EVALUATE
############################################################

uniform8_model.eval()

y_true = []
y_pred = []

with torch.no_grad():

    for x, y in test_loader:

        x = x.to(device)

        outputs = uniform8_model(x)

        preds = outputs.argmax(1)

        y_true.extend(y.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

acc = 100 * np.mean(
    np.array(y_true) ==
    np.array(y_pred)
)

print("="*60)
print("UNIFORM 8-BIT QUANTIZATION")
print("="*60)
print(f"Accuracy = {acc:.4f}%")

print("\nConfusion Matrix")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report")
print(classification_report(y_true, y_pred))

UNIFORM 8-BIT QUANTIZATION
Accuracy = 93.5238%

Confusion Matrix
[[522   1   0   0   0   2]
 [  0 499   2   0   4  20]
 [  0   2 514   2   0   7]
 [  2   0   6 474  35   8]
 [  0  11   2  28 444  40]
 [  1   7   2   2  20 493]]

Classification Report
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       525
           1       0.96      0.95      0.96       525
           2       0.98      0.98      0.98       525
           3       0.94      0.90      0.92       525
           4       0.88      0.85      0.86       525
           5       0.86      0.94      0.90       525

    accuracy                           0.94      3150
   macro avg       0.94      0.94      0.94      3150
weighted avg       0.94      0.94      0.94      3150



In [12]:
############################################################
# UNIFORM 6-BIT QUANTIZATION
############################################################

import copy
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

base_model = PCA_CNN().to(device)

base_model.load_state_dict(
    torch.load("Models/best_model_depthwise.pth")
)

base_model.eval()

uniform6_model = copy.deepcopy(base_model)

with torch.no_grad():

    for name, param in uniform6_model.named_parameters():

        if "weight" in name:

            param.data = quantize_weights(
                param.data,
                bits=6
            )

uniform6_model.eval()

y_true = []
y_pred = []

with torch.no_grad():

    for x, y in test_loader:

        x = x.to(device)

        outputs = uniform6_model(x)

        preds = outputs.argmax(1)

        y_true.extend(y.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

acc = 100*np.mean(
    np.array(y_true)==
    np.array(y_pred)
)

print("="*60)
print("UNIFORM 6-BIT QUANTIZATION")
print("="*60)
print(f"Accuracy = {acc:.4f}%")

print("\nConfusion Matrix")
print(confusion_matrix(y_true,y_pred))

print("\nClassification Report")
print(classification_report(y_true,y_pred))

UNIFORM 6-BIT QUANTIZATION
Accuracy = 92.9524%

Confusion Matrix
[[522   1   0   0   0   2]
 [  0 499   1   0  11  14]
 [  0   2 513   2   0   8]
 [  2   0   6 485  26   6]
 [  0   9   2  54 429  31]
 [  2  16   0   2  25 480]]

Classification Report
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       525
           1       0.95      0.95      0.95       525
           2       0.98      0.98      0.98       525
           3       0.89      0.92      0.91       525
           4       0.87      0.82      0.84       525
           5       0.89      0.91      0.90       525

    accuracy                           0.93      3150
   macro avg       0.93      0.93      0.93      3150
weighted avg       0.93      0.93      0.93      3150



In [13]:
############################################################
# LAYER-WISE QUANTIZATION SENSITIVITY
############################################################

import copy



def evaluate_model(model):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for x, y in test_loader:

            x = x.to(device)
            y = y.to(device)

            outputs = model(x)

            preds = outputs.argmax(1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    return 100.0 * correct / total


############################################################
# LOAD BEST MODEL
############################################################

base_model = PCA_CNN().to(device)

base_model.load_state_dict(
    torch.load("Models/best_model_depthwise.pth")
)

baseline_acc = evaluate_model(base_model)

print(f"\nBaseline Accuracy = {baseline_acc:.4f}%")


Baseline Accuracy = 93.5238%


In [14]:
############################################################
# TEST EACH LAYER
############################################################

layers_to_test = [
    "dw1.weight",
    "pw1.weight",
    "dw2.weight",
    "pw2.weight",
    "fc1.weight",
    "fc2.weight"
]

bitwidths = [8, 6, 4, 2]

results = {}

for layer_name in layers_to_test:

    results[layer_name] = {}

    print("\n" + "="*60)
    print("Testing:", layer_name)

    for bits in bitwidths:

        model = copy.deepcopy(base_model)

        with torch.no_grad():

            for name, param in model.named_parameters():

                if name == layer_name:

                    param.data = quantize_weights(
                        param.data,
                        bits
                    )

        acc = evaluate_model(model)

        results[layer_name][bits] = acc

        print(
            f"{bits:2d}-bit --> {acc:.4f}%"
        )


Testing: dw1.weight
 8-bit --> 93.4603%
 6-bit --> 93.5238%
 4-bit --> 93.7143%
 2-bit --> 92.5079%

Testing: pw1.weight
 8-bit --> 93.4921%
 6-bit --> 93.5556%
 4-bit --> 93.2381%
 2-bit --> 80.0000%

Testing: dw2.weight
 8-bit --> 93.5556%
 6-bit --> 93.4603%
 4-bit --> 93.5556%
 2-bit --> 91.3968%

Testing: pw2.weight
 8-bit --> 93.5238%
 6-bit --> 93.5556%
 4-bit --> 92.8571%
 2-bit --> 48.9841%

Testing: fc1.weight
 8-bit --> 93.4603%
 6-bit --> 93.5238%
 4-bit --> 88.0635%
 2-bit --> 35.3651%

Testing: fc2.weight
 8-bit --> 93.3651%
 6-bit --> 92.5079%
 4-bit --> 76.6032%
 2-bit --> 33.0794%


In [15]:
############################################################
# PRINT TABLE
############################################################

print("\n")
print("="*70)
print("HAQ SENSITIVITY TABLE")
print("="*70)

for layer, vals in results.items():

    print(
        f"{layer:15s}",
        end=""
    )

    for bits in [8,6,4,2]:

        print(
            f"{vals[bits]:8.2f}",
            end=""
        )

    print()

print("\nColumns = [8bit 6bit 4bit 2bit]")



HAQ SENSITIVITY TABLE
dw1.weight        93.46   93.52   93.71   92.51
pw1.weight        93.49   93.56   93.24   80.00
dw2.weight        93.56   93.46   93.56   91.40
pw2.weight        93.52   93.56   92.86   48.98
fc1.weight        93.46   93.52   88.06   35.37
fc2.weight        93.37   92.51   76.60   33.08

Columns = [8bit 6bit 4bit 2bit]


In [16]:
############################################################
# EXHAUSTIVE MIXED-PRECISION SEARCH
############################################################

import copy
import itertools
import pandas as pd

layers = [
    "dw1.weight",
    "pw1.weight",
    "dw2.weight",
    "pw2.weight",
    "fc1.weight",
    "fc2.weight"
]

bit_choices = [8, 6, 4]

results = []

total = len(bit_choices) ** len(layers)

count = 1

for bits in itertools.product(bit_choices, repeat=6):

    print(f"Testing {count}/{total} : {bits}")

    model = copy.deepcopy(base_model)

    ########################################################
    # QUANTIZE SELECTED LAYERS
    ########################################################

    with torch.no_grad():

        for name, param in model.named_parameters():

            if name in layers:

                idx = layers.index(name)

                param.data = quantize_weights(
                    param.data,
                    bits[idx]
                )

    ########################################################
    # EVALUATE
    ########################################################

    acc = evaluate_model(model)

    ########################################################
    # DEBUG FOR UNIFORM 8-BIT
    ########################################################

    if bits == (8, 8, 8, 8, 8, 8):

        print("\nUniform 8-bit Accuracy =", acc)

        print("\nWeight Sums:")

        for name, param in model.named_parameters():

            if "weight" in name:

                print(name, torch.sum(param).item())

    ########################################################
    # SAVE RESULT
    ########################################################

    results.append({
        "DW1": bits[0],
        "PW1": bits[1],
        "DW2": bits[2],
        "PW2": bits[3],
        "FC1": bits[4],
        "FC2": bits[5],
        "Accuracy": acc
    })

    count += 1


############################################################
# CREATE DATAFRAME
############################################################

results_df = pd.DataFrame(results)

print("\nNumber of Results :", len(results_df))

print("\nUniform 8-bit Configuration:")

print(
    results_df[
        (results_df["DW1"] == 8) &
        (results_df["PW1"] == 8) &
        (results_df["DW2"] == 8) &
        (results_df["PW2"] == 8) &
        (results_df["FC1"] == 8) &
        (results_df["FC2"] == 8)
    ]
)

print("\nTop 10 Configurations:")

print(
    results_df.sort_values(
        by="Accuracy",
        ascending=False
    ).head(10)
)

Testing 1/729 : (8, 8, 8, 8, 8, 8)

Uniform 8-bit Accuracy = 93.52380952380952

Weight Sums:
dw1.weight 0.5915051698684692
pw1.weight 1.0158567428588867
dw2.weight 3.8860244750976562
pw2.weight -18.434661865234375
fc1.weight -107.2735595703125
fc2.weight -104.07462310791016
Testing 2/729 : (8, 8, 8, 8, 8, 6)
Testing 3/729 : (8, 8, 8, 8, 8, 4)
Testing 4/729 : (8, 8, 8, 8, 6, 8)
Testing 5/729 : (8, 8, 8, 8, 6, 6)
Testing 6/729 : (8, 8, 8, 8, 6, 4)
Testing 7/729 : (8, 8, 8, 8, 4, 8)
Testing 8/729 : (8, 8, 8, 8, 4, 6)
Testing 9/729 : (8, 8, 8, 8, 4, 4)
Testing 10/729 : (8, 8, 8, 6, 8, 8)
Testing 11/729 : (8, 8, 8, 6, 8, 6)
Testing 12/729 : (8, 8, 8, 6, 8, 4)
Testing 13/729 : (8, 8, 8, 6, 6, 8)
Testing 14/729 : (8, 8, 8, 6, 6, 6)
Testing 15/729 : (8, 8, 8, 6, 6, 4)
Testing 16/729 : (8, 8, 8, 6, 4, 8)
Testing 17/729 : (8, 8, 8, 6, 4, 6)
Testing 18/729 : (8, 8, 8, 6, 4, 4)
Testing 19/729 : (8, 8, 8, 4, 8, 8)
Testing 20/729 : (8, 8, 8, 4, 8, 6)
Testing 21/729 : (8, 8, 8, 4, 8, 4)
Testing 22/72

In [17]:
for name, param in base_model.named_parameters():
    print(name)

dw1.weight
dw1.bias
pw1.weight
pw1.bias
dw2.weight
dw2.bias
pw2.weight
pw2.bias
fc1.weight
fc1.bias
fc2.weight
fc2.bias


In [18]:
############################################################
# CREATE TABLE
############################################################

results_df = pd.DataFrame(
    results,
    columns=[
    "DW1",
    "PW1",
    "DW2",
    "PW2",
    "FC1",
    "FC2",
    "Accuracy"
]
)

results_df = results_df.sort_values(
    by="Accuracy",
    ascending=False
)

results_df = results_df.reset_index(drop=True)

print(results_df)

     DW1  PW1  DW2  PW2  FC1  FC2   Accuracy
0      4    8    6    8    8    8  93.746032
1      6    6    6    6    8    8  93.650794
2      6    8    8    8    8    8  93.650794
3      6    8    6    6    8    8  93.619048
4      6    8    6    6    6    8  93.619048
..   ...  ...  ...  ...  ...  ...        ...
724    4    8    4    4    4    4  61.968254
725    8    6    4    4    4    4  61.841270
726    6    6    4    4    4    4  61.809524
727    8    8    4    4    4    4  61.777778
728    6    8    4    4    4    4  61.746032

[729 rows x 7 columns]


In [19]:
results_df.to_csv(
    "Mixed_Quantization_Results.csv",
    index=False
)

print("Saved Successfully")

Saved Successfully


In [20]:
############################################################
# HARDWARE COST ANALYSIS
############################################################

import numpy as np

PARAMS = {
    "DW1": 18,
    "PW1": 224,
    "DW2": 96,
    "PW2": 1056,
    "FC1": 2112,
    "FC2": 390
}

# FP32 reference
FP32_BITS = (
    (
        PARAMS["DW1"] +
        PARAMS["PW1"] +
        PARAMS["DW2"] +
        PARAMS["PW2"] +
        PARAMS["FC1"] +
        PARAMS["FC2"]
    ) * 32
)

# Calculate hardware cost
results_df["Memory(bits)"] = (
    results_df["DW1"] * PARAMS["DW1"] +
    results_df["PW1"] * PARAMS["PW1"] +
    results_df["DW2"] * PARAMS["DW2"] +
    results_df["PW2"] * PARAMS["PW2"] +
    results_df["FC1"] * PARAMS["FC1"] +
    results_df["FC2"] * PARAMS["FC2"]
)

results_df["Memory(KB)"] = (
    results_df["Memory(bits)"] / 8 / 1024
)

results_df["Compression"] = (
    FP32_BITS / results_df["Memory(bits)"]
)

# Xilinx BRAM18 = 18 Kbits = 18432 bits
results_df["BRAM18"] = np.ceil(
    results_df["Memory(bits)"] / 18432
).astype(int)

############################################################
# KEEP ONLY GOOD MODELS
############################################################

good_models = results_df[
    results_df["Accuracy"] >= 93.0
].copy()

good_models = good_models.sort_values(
    by=["Accuracy", "Memory(bits)"],
    ascending=[False, True]
)

good_models = good_models.reset_index(drop=True)

print(good_models)

     DW1  PW1  DW2  PW2  FC1  FC2   Accuracy  Memory(bits)  Memory(KB)  \
0      4    8    6    8    8    8  93.746032         30904    3.772461   
1      6    6    6    6    8    8  93.650794         28380    3.464355   
2      6    8    8    8    8    8  93.650794         31132    3.800293   
3      6    8    6    6    6    8  93.619048         24604    3.003418   
4      6    8    6    6    8    8  93.619048         28828    3.519043   
..   ...  ...  ...  ...  ...  ...        ...           ...         ...   
112    4    8    8    6    8    6  93.047619         28204    3.442871   
113    6    8    8    6    8    6  93.047619         28240    3.447266   
114    8    6    6    6    6    6  93.015873         23412    2.857910   
115    6    8    6    6    6    6  93.015873         23824    2.908203   
116    6    8    6    6    8    6  93.015873         28048    3.423828   

     Compression  BRAM18  
0       4.034170       2  
1       4.392953       2  
2       4.004625       2  
3  

In [21]:
############################################################
# SAVE RESULTS
############################################################

good_models.to_csv(
    "Hardware_Aware_Results.csv",
    index=False
)

print("Saved Successfully!")

print("\nTop Hardware-Aware Models\n")
print(good_models)

Saved Successfully!

Top Hardware-Aware Models

     DW1  PW1  DW2  PW2  FC1  FC2   Accuracy  Memory(bits)  Memory(KB)  \
0      4    8    6    8    8    8  93.746032         30904    3.772461   
1      6    6    6    6    8    8  93.650794         28380    3.464355   
2      6    8    8    8    8    8  93.650794         31132    3.800293   
3      6    8    6    6    6    8  93.619048         24604    3.003418   
4      6    8    6    6    8    8  93.619048         28828    3.519043   
..   ...  ...  ...  ...  ...  ...        ...           ...         ...   
112    4    8    8    6    8    6  93.047619         28204    3.442871   
113    6    8    8    6    8    6  93.047619         28240    3.447266   
114    8    6    6    6    6    6  93.015873         23412    2.857910   
115    6    8    6    6    6    6  93.015873         23824    2.908203   
116    6    8    6    6    8    6  93.015873         28048    3.423828   

     Compression  BRAM18  
0       4.034170       2  
1       4

In [22]:
print("Depthwise Conv1 :", model.dw1.weight.numel())
print("Pointwise Conv1 :", model.pw1.weight.numel())

print("Depthwise Conv2 :", model.dw2.weight.numel())
print("Pointwise Conv2 :", model.pw2.weight.numel())

print("FC1 :", model.fc1.weight.numel())
print("FC2 :", model.fc2.weight.numel())

print("\nTotal Parameters:",
      sum(p.numel() for p in model.parameters()))

Depthwise Conv1 : 12
Pointwise Conv1 : 192
Depthwise Conv2 : 64
Pointwise Conv2 : 1024
FC1 : 2048
FC2 : 384

Total Parameters: 3896
